In [ ]:
%pip install pytesseract

In [ ]:
import cv2
import numpy as np
import pytesseract
import re
import os
from sklearn.cluster import KMeans
from PIL import Image

ModuleNotFoundError: No module named 'pytesseract'

In [ ]:
def ocr_tesseract(image, psm=7):
    """Run Tesseract OCR on a small region (optimized for single character)."""
    config = f'--psm {psm} -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    text = pytesseract.image_to_string(image, config=config)
    text = text.strip().upper()
    return text if len(text) == 1 and text.isalpha() else None

In [ ]:
def parse_strands_screenshot(path="strands_screenshot.png"):
    """
    Tesseract-based version of the hybrid vision + OCR pipeline for NYT Strands.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find the screenshot file: {path}")

    print(f"\n--- Processing {os.path.basename(path)} ---")
    img = cv2.imread(path)
    if img is None:
        raise IOError(f"Cannot read the image file: {path}")

    h, w, _ = img.shape

    # ========== 1. Extract THEME ==========
    print("Step 1: Reading theme from left side...")
    left_crop = img[:, :int(w / 2)]
    gray_left = cv2.cvtColor(left_crop, cv2.COLOR_BGR2GRAY)
    theme_text = pytesseract.image_to_string(gray_left, config='--psm 6').strip()

    theme = "Theme not found"
    word_count = None

    # Try to find line after "Theme"
    theme_match = re.search(r"THEME\s*[\:\-]?\s*(.+)", theme_text, re.IGNORECASE)
    if theme_match:
        theme = theme_match.group(1).split("\n")[0].strip()

    wc_match = re.search(r"of\s+(\d+)\s+theme", theme_text, re.IGNORECASE)
    if wc_match:
        word_count = int(wc_match.group(1))

    print(f"Theme: '{theme}', Word Count: {word_count}")

    # ========== 2. Extract GRID using Adaptive Contours ==========
    print("Step 2: Processing grid with adaptive filters...")
    grid_crop = img[:, int(w / 2):]
    gray = cv2.cvtColor(grid_crop, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)

    # Adaptive threshold to isolate letters
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 21, 10
    )

    # Morphological smoothing
    kernel = np.ones((2, 2), np.uint8)
    morphed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        raise ValueError("No contours detected in grid region.")

    areas = [cv2.contourArea(c) for c in contours if cv2.contourArea(c) > 20]
    median_area = np.median(areas)
    if median_area <= 0:
        raise ValueError("Invalid median contour area.")

    letter_boxes = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if median_area * 0.2 < area < median_area * 3.5:
            x, y, w_cnt, h_cnt = cv2.boundingRect(cnt)
            roi = gray[y:y + h_cnt, x:x + w_cnt]
            roi = cv2.copyMakeBorder(roi, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=255)

            char = ocr_tesseract(roi)
            if char:
                cx, cy = x + w_cnt / 2, y + h_cnt / 2
                letter_boxes.append({'char': char, 'cx': cx, 'cy': cy})

    print(f"Detected {len(letter_boxes)} valid letters after filtering.")

    # ========== 3. Grid Assembly ==========
    print("Step 3: Assembling 8x6 grid...")
    grid = [["?" for _ in range(6)] for _ in range(8)]

    if len(letter_boxes) >= 25:
        y_coords = np.array([b['cy'] for b in letter_boxes]).reshape(-1, 1)
        x_coords = np.array([b['cx'] for b in letter_boxes]).reshape(-1, 1)

        kmeans_y = KMeans(n_clusters=8, random_state=0, n_init='auto').fit(y_coords)
        kmeans_x = KMeans(n_clusters=6, random_state=0, n_init='auto').fit(x_coords)

        row_centers = sorted(kmeans_y.cluster_centers_.flatten())
        col_centers = sorted(kmeans_x.cluster_centers_.flatten())

        for b in letter_boxes:
            r = np.argmin([abs(b['cy'] - ry) for ry in row_centers])
            c = np.argmin([abs(b['cx'] - cx) for cx in col_centers])
            grid[r][c] = b['char']

    filled = sum(1 for row in grid for c in row if c != "?")
    print(f"Grid filled: {filled}/48")

    return {"theme": theme, "grid": grid, "word_count": word_count}

In [ ]:
def print_results(data):
    """Pretty-print results."""
    print("\n--- FINAL OCR RESULTS ---")
    print("{")
    print(f"  'theme': '{data['theme']}',")
    print("  'grid': [")
    for i, row in enumerate(data['grid']):
        row_str = ", ".join([f"'{c}'" for c in row])
        comma = "," if i < len(data['grid']) - 1 else ""
        print(f"    [{row_str}]{comma}")
    print("  ],")
    print("  'grid_coordinates': [")
    for i, row in enumerate(data['grid_coordinates']):
        row_str = ", ".join([f"({x}, {y})" if x is not None and y is not None else "null" for x, y in row])
        comma = "," if i < len(data['grid_coordinates']) - 1 else ""
        print(f"    [{row_str}]{comma}")
    print("  ],")
    print(f"  'word_count': {data.get('word_count', 'N/A')}")
    print("}")


if __name__ == "__main__":
    try:
        results = parse_strands_screenshot("strands_screenshot.png")
        print_results(results)
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()


Error: name 'os' is not defined


Traceback (most recent call last):
  File "/tmp/ipython-input-672035289.py", line 24, in <cell line: 0>
    results = parse_strands_screenshot("coordinate_debug.png")
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipython-input-1720759507.py", line 5, in parse_strands_screenshot
    if not os.path.exists(path):
           ^^
NameError: name 'os' is not defined. Did you forget to import 'os'


#GRID COORDS EXTRACTION ---


In [ ]:
# Install/Setup commands (kept for context, but will execute outside the class scope)
%pip install pytesseract

In [ ]:
import cv2
import numpy as np
import pytesseract
import re
import os
from sklearn.cluster import KMeans
from PIL import Image

In [ ]:
def ocr_tesseract(image, psm=7):
    """Run Tesseract OCR on a small region (optimized for single character)."""
    # Whitelist all uppercase letters
    config = f'--psm {psm} -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    text = pytesseract.image_to_string(image, config=config)
    text = text.strip().upper()
    return text if len(text) == 1 and text.isalpha() else None

In [ ]:
def parse_strands_screenshot(path="coordinate_debug.png"):
    """
    Tesseract-based version of the hybrid vision + OCR pipeline for NYT Strands.
    Includes logic to find and map the pixel coordinates of each letter cell.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find the screenshot file: {path}")

    print(f"\n--- Processing {os.path.basename(path)} ---")
    img = cv2.imread(path)
    if img is None:
        raise IOError(f"Cannot read the image file: {path}")

    h, w, _ = img.shape

    # Calculate the X-offset for the grid area relative to the full image
    x_offset = int(w / 2)

    # ========== 1. Extract THEME ==========
    print("Step 1: Reading theme from left side...")
    left_crop = img[:, :x_offset]
    gray_left = cv2.cvtColor(left_crop, cv2.COLOR_BGR2GRAY)

    # Use psm 6 for reading block text
    theme_text = pytesseract.image_to_string(gray_left, config='--psm 6').strip()

    theme = "Theme not found"
    word_count = None

    # Try to find line after "Theme"
    theme_match = re.search(r"THEME\s*[\:\-]?\s*(.+)", theme_text, re.IGNORECASE)
    if theme_match:
        theme = theme_match.group(1).split("\n")[0].strip()

    wc_match = re.search(r"of\s+(\d+)\s+theme", theme_text, re.IGNORECASE)
    if wc_match:
        word_count = int(wc_match.group(1))

    print(f"Theme: '{theme}', Word Count: {word_count}")

    # ========== 2. Extract GRID using Adaptive Contours ==========
    print("Step 2: Processing grid with adaptive filters...")
    grid_crop = img[:, x_offset:]
    gray = cv2.cvtColor(grid_crop, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)

    # Adaptive threshold to isolate letters
    thresh = cv2.adaptiveThreshold(
        blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 21, 10
    )

    # Morphological smoothing
    kernel = np.ones((2, 2), np.uint8)
    morphed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(morphed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        raise ValueError("No contours detected in grid region.")

    areas = [cv2.contourArea(c) for c in contours if cv2.contourArea(c) > 20]
    if not areas:
        raise ValueError("No significant contours detected in grid region.")

    median_area = np.median(areas)
    if median_area <= 0:
        raise ValueError("Invalid median contour area.")

    letter_boxes = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        # Filter contours based on size, typical letter size is roughly median area
        if median_area * 0.2 < area < median_area * 3.5:
            x, y, w_cnt, h_cnt = cv2.boundingRect(cnt)

            # Create ROI for OCR
            roi = gray[y:y + h_cnt, x:x + w_cnt]
            # Add border for better Tesseract performance
            roi = cv2.copyMakeBorder(roi, 10, 10, 10, 10, cv2.BORDER_CONSTANT, value=255)

            char = ocr_tesseract(roi)
            if char:
                # cx, cy are center coordinates relative to the cropped grid image (grid_crop)
                cx, cy = x + w_cnt / 2, y + h_cnt / 2
                letter_boxes.append({'char': char, 'cx': cx, 'cy': cy})

    print(f"Detected {len(letter_boxes)} valid letters after filtering.")

    # ========== 3. Grid Assembly and Coordinate Mapping ==========
    print("Step 3: Assembling 8x6 grid and mapping coordinates...")
    grid = [["?" for _ in range(6)] for _ in range(8)]
    # Initialize a parallel structure to store (x, y) coordinates
    grid_coordinates = [[(None, None) for _ in range(6)] for _ in range(8)]

    if len(letter_boxes) >= 25: # Need a reasonable number of letters to cluster reliably
        y_coords = np.array([b['cy'] for b in letter_boxes]).reshape(-1, 1)
        x_coords = np.array([b['cx'] for b in letter_boxes]).reshape(-1, 1)

        # Cluster coordinates to find stable row and column centers
        kmeans_y = KMeans(n_clusters=8, random_state=0, n_init='auto').fit(y_coords)
        kmeans_x = KMeans(n_clusters=6, random_state=0, n_init='auto').fit(x_coords)

        row_centers = sorted(kmeans_y.cluster_centers_.flatten())
        col_centers = sorted(kmeans_x.cluster_centers_.flatten())

        for b in letter_boxes:
            # Determine the closest grid cell (r, c)
            r = np.argmin([abs(b['cy'] - ry) for ry in row_centers])
            c = np.argmin([abs(b['cx'] - cx) for cx in col_centers])

            # 1. Store character
            grid[r][c] = b['char']

            # 2. Store coordinates relative to the original full image
            # X coordinate must be offset by the cropped amount (x_offset)
            full_x = int(b['cx'] + x_offset)
            full_y = int(b['cy'])
            grid_coordinates[r][c] = (full_x, full_y)

    filled = sum(1 for row in grid for c in row if c != "?")
    print(f"Grid filled: {filled}/48")

    return {"theme": theme, "grid": grid, "word_count": word_count, "grid_coordinates": grid_coordinates}

In [ ]:
def print_results(data):
    """Pretty-print results."""
    print("\n--- FINAL OCR RESULTS ---")
    print("{")
    print(f"  'theme': '{data['theme']}',")
    print("  'grid': [")
    for i, row in enumerate(data['grid']):
        row_str = ", ".join([f"'{c}'" for c in row])
        comma = "," if i < len(data['grid']) - 1 else ""
        print(f"    [{row_str}]{comma}")
    print("  ],")
    print("  'grid_coordinates': [")
    # Correctly iterate and print the coordinates
    for i, row in enumerate(data['grid_coordinates']):
        # Format coordinates as (x, y) or 'null' if not found
        row_str = ", ".join([f"({x}, {y})" if x is not None else "null" for x, y in row])
        comma = "," if i < len(data['grid_coordinates']) - 1 else ""
        print(f"    [{row_str}]{comma}")
    print("  ],")
    print(f"  'word_count': {data.get('word_count', 'N/A')}")
    print("}")

In [ ]:
if __name__ == "__main__":
    try:
        results = parse_strands_screenshot("coordinate_debug.png")
        print_results(results)
    except Exception as e:
        print(f"\nError: {e}")
        import traceback
        traceback.print_exc()


--- Processing coordinate_debug.png ---
Step 1: Reading theme from left side...
Theme: 'Catch all', Word Count: 7
Step 2: Processing grid with adaptive filters...
Detected 48 valid letters after filtering.
Step 3: Assembling 8x6 grid and mapping coordinates...
Grid filled: 48/48

--- FINAL OCR RESULTS ---
{
  'theme': 'Catch all',
  'grid': [
    ['T', 'E', 'R', 'I', 'N', 'I'],
    ['B', 'T', 'Y', 'C', 'L', 'O'],
    ['A', 'E', 'N', 'E', 'P', 'C'],
    ['O', 'L', 'U', 'H', 'T', 'T'],
    ['H', 'S', 'A', 'M', 'B', 'A'],
    ['J', 'C', 'A', 'P', 'C', 'R'],
    ['U', 'E', 'T', 'E', 'K', 'E'],
    ['N', 'K', 'D', 'R', 'A', 'W']
  ],
  'grid_coordinates': [
    [(1052, 215), (1109, 215), (1164, 215), (1219, 215), (1275, 215), (1331, 215)],
    [(1053, 269), (1108, 269), (1164, 269), (1220, 269), (1276, 269), (1331, 269)],
    [(1052, 323), (1109, 323), (1163, 323), (1220, 323), (1276, 323), (1331, 323)],
    [(1052, 377), (1109, 377), (1164, 377), (1219, 377), (1275, 377), (1331, 377)],
  